# Analyse des résultats d'évaluation

Ce notebook charge les fichiers `outputs/*/results.json` (format avec clés `experiment` et `results`), calcule des moyennes par type de test et trace une visualisation simple.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("..").resolve()
paths = sorted((ROOT / "outputs").glob("*/results.json"))
print("Fichiers:", [p.relative_to(ROOT).as_posix() for p in paths])

In [ ]:
def load_document(path: Path):
    with open(path, encoding="utf-8") as f:
        doc = json.load(f)
    if isinstance(doc, dict) and "results" in doc:
        return doc["experiment"], doc["results"]
    return None, doc


def mean_score_by_type(rows):
    sums, counts = {}, {}
    for row in rows:
        t = row.get("prompt_type", row.get("type", "?"))
        sc = row.get("scores", row)
        sg = float(sc.get("score_global", row.get("score_global", 0)))
        sums[t] = sums.get(t, 0.0) + sg
        counts[t] = counts.get(t, 0) + 1
    return {t: sums[t] / counts[t] for t in sums}


series = []
for p in paths:
    exp, rows = load_document(p)
    label = exp["model_name"] if exp else p.parent.name
    series.append((label, mean_score_by_type(rows)))
    print(label, series[-1][1])

In [ ]:
if not series:
    print("Aucun résultat : exécutez src/evaluate.py d'abord.")
else:
    types = sorted({k for _, m in series for k in m.keys()})
    x = np.arange(len(types))
    width = 0.8 / max(len(series), 1)
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, (label, means) in enumerate(series):
        vals = [means.get(t, float("nan")) for t in types]
        ax.bar(x + i * width, vals, width, label=label)
    ax.set_xticks(x + width * (len(series) - 1) / 2)
    ax.set_xticklabels(types)
    ax.set_ylabel("Score moyen (score_global)")
    ax.set_title("Comparaison par type de test")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()